# 07 — Interactive Drone CAD Viewer

This notebook launches a live 3D viewer for the drone CAD model. Every slider rebuilds the geometry via the same `build_drone_parts` function the STL exporter uses — so what you see is exactly what you'd print.

- **Global parameters** (plate, sphere, motor disc) live in the collapsible *Global parameters* panel and apply to every part that uses them.
- **Per-arm parameters** (tilt, azimuth, length, attachment angle) live in one tab per arm.
- A full rebuild takes ~3–5 seconds. Untick **Auto-rebuild** if you want to change several sliders before waiting for a rebuild — then click **Rebuild**.

## Dependencies

Requires `ipywidgets>=8` and `jupyter-cadquery>=3.7` (already listed in `requirements.txt`). If you get an `ImportError` here, run `pip install -r ../requirements.txt`.

In [1]:
import sys
sys.path.insert(0, '..')

from drone.cad import interactive_viewer

# No args → loads genomes/hexacopter.py (6 arms evenly spaced).
# Pass a numpy array of shape (n, 6) or a genome handler to use your own design.
interactive_viewer()

## Tips

- **Arm length floor**: the build pipeline needs `arm_length` ≥ `plate_radius + (sphere_radius - clamp_inset) + motor_mount_cylinder_height + sphere_offset`. With defaults that's ~56.5 mm, so the slider starts at 60 mm. If you push it below the feasible limit the status line shows the error and keeps the previous render on screen.
- **Plate changes move the arms**: when you change `plate_diameter` or `outer_ring_width`, the attachment radial distance updates so the arms stay clamped to the (new) rim.
- **Loading a different genome**:

    ```python
    from drone.genome import DroneGenome
    g = DroneGenome.load('../genomes/evolved_example.py')
    interactive_viewer(g.data)
    ```

- **Exporting the current state**: once you're happy with the widget values, read them off the returned `VBox` children and pass them as `core_plate_overrides` / `part_config` to `generate_stl_files`.

## Single-part viewer

Pick a part from the dropdown and only the sliders relevant to that part appear. Useful for tuning one part in isolation without recomputing the whole drone on every slider tick.

- **Central plate**: the core plate.
- **Arm mount**: just the sphere-with-socket.
- **Motor mount**: just the motor_arm (hollow socket + disc).
- **Full arm fused**: the solid silhouette used for the per-arm fused STL. Genome-side `arm_length` (drone centre → disc), so the ~56.5 mm feasibility floor applies.

The camera re-frames to the selected part on every rebuild, so switching between a 60 mm plate and a 180 mm fused arm always fills the viewport.

In [2]:
from drone.cad import interactive_part_viewer

interactive_part_viewer()